# Notebook 11: Pytorch Lightning
In this notebook, we will see how to use pytorch lightning to organize our pytorch code, writing a network to classify MNIST images as an example (reminder: we did this in notebook 5) 

To add a demo of another cool thing, we'll use **tensorboard** to inspect our model's training

In [2]:
!pip install -q lightning

In [3]:
# Re-using the code from previous notebooks
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
# new imports
import torch.nn.functional as F
import lightning as L

# creating MNIST datasets and dataloaders as before
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

100%|██████████████████████████████████████| 9.91M/9.91M [00:00<00:00, 12.8MB/s]
100%|███████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 313kB/s]
100%|██████████████████████████████████████| 1.65M/1.65M [00:00<00:00, 3.55MB/s]
100%|██████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 15.0MB/s]


In [5]:
# New: our lightning module!

class LitNeuralNetwork(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=0.001)
    
    def training_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = F.cross_entropy(logits, y)
        self.log('train_loss', loss, on_epoch=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = F.cross_entropy(logits, y)
        correct = (logits.argmax(1) == y).float().sum()
        correct /= len(y)
        self.log('val_loss', loss)
        self.log('Accuracy', correct)
        return loss


In [7]:
# for visualization during training: tensorboard
!pip install -q tensorboard
%load_ext tensorboard
%tensorboard --logdir .

In [8]:
# instantiating model and trainer, and training the model

callbacks = None
model = LitNeuralNetwork()
trainer = L.Trainer(max_epochs=10, 
                    callbacks=callbacks,
                    accelerator="cpu", devices=1)
trainer.fit(model, train_dataloader, test_dataloader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name              | Type       | Params | Mode  | FLOPs
-----------------------------------------------------------------
0 | flatten           | Flatten    | 0      | train | 0    
1 | linear_relu_stack | Sequential | 118 K  | train | 0    
-----------------------------------------------------------------
118 K     Trainable params
0         Non-trainable params
118 K     Total params
0.473     Total estimated model params size (MB)
7         Modules in train mode
0         Modul

Sanity Checking: |                                        | 0/? [00:00<?, ?it/s]

/home/giuliafazzino/.local/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/home/giuliafazzino/.local/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.


Training: |                                               | 0/? [00:00<?, ?it/s]

Validation: |                                             | 0/? [00:00<?, ?it/s]

...

Validation: |                                             | 0/? [00:00<?, ?it/s]

Validation: |                                             | 0/? [00:00<?, ?it/s]

Validation: |                                             | 0/? [00:00<?, ?it/s]

Validation: |                                             | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
